# Week 9 preliminary results

This notebook presents the Week 9 checkpoint from the registered OWMAS experiment. It reads the fixed outputs under `results/`; it does not rerun the backtest or change any registered setting.

The checkpoint covers the full-sample performance table, Sharpe-ratio significance tests, equity curves, and drawdowns.

In [ ]:
from pathlib import Path
import json

import matplotlib.pyplot as plt
import pandas as pd

ROOT = Path.cwd()
if not (ROOT / 'results').is_dir():
    ROOT = ROOT.parent
RESULTS = ROOT / 'results'
required = [
    'run_metadata.json',
    'performance_metrics.csv',
    'h1_sharpe_tests.csv',
    'equity_curves.csv',
    'drawdowns.csv',
]
missing = [name for name in required if not (RESULTS / name).is_file()]
assert not missing, f'Missing registered outputs: {missing}. Run scripts/run_experiment.py first.'
print(f'Reading registered outputs from: {RESULTS.resolve()}')

## Registered run

In [ ]:
metadata = json.loads((RESULTS / 'run_metadata.json').read_text(encoding='utf-8'))
run_details = pd.Series({
    'Sample': f"{metadata['start']} to {metadata['end']}",
    'Trading days': metadata['observations'],
    'Bootstrap replications': metadata['bootstrap_reps'],
    'Confidence level': f"{metadata['confidence']:.0%}",
    'Random seed': metadata['seed'],
})
print(run_details.to_string())

## Main performance table

Returns, drawdowns, volatility, and exposure are shown as percentages. Sharpe and Sortino ratios are annualised.

In [ ]:
performance = pd.read_csv(RESULTS / 'performance_metrics.csv').set_index('strategy')
performance_table = performance.rename(columns={
    'cumulative_return': 'Cumulative return',
    'cagr': 'CAGR',
    'sharpe': 'Sharpe',
    'sortino': 'Sortino',
    'max_drawdown': 'Max drawdown',
    'volatility': 'Volatility',
    'exposure': 'Exposure',
    'trades': 'Trades',
})[[
    'Cumulative return', 'CAGR', 'Sharpe', 'Sortino',
    'Max drawdown', 'Volatility', 'Exposure', 'Trades'
]]
formatters = {
    'Cumulative return': lambda value: f'{value:.2%}',
    'CAGR': lambda value: f'{value:.2%}',
    'Sharpe': lambda value: f'{value:.3f}',
    'Sortino': lambda value: f'{value:.3f}',
    'Max drawdown': lambda value: f'{value:.2%}',
    'Volatility': lambda value: f'{value:.2%}',
    'Exposure': lambda value: f'{value:.2%}',
    'Trades': lambda value: f'{int(value)}',
}
print(performance_table.to_string(formatters=formatters))

In [ ]:
owmas = performance.loc['OWMAS']
print(
    f"OWMAS returned {owmas['cumulative_return']:.2%} cumulatively, "
    f"with a {owmas['sharpe']:.3f} Sharpe ratio and "
    f"a {owmas['max_drawdown']:.2%} maximum drawdown."
)
best_sharpe = performance['sharpe'].idxmax()
print(f'The highest full-sample Sharpe point estimate was {best_sharpe}: {performance.loc[best_sharpe, "sharpe"]:.3f}.')

## Sharpe-ratio significance tests

The table reports paired two-sided Ledoit-Wolf tests with 95% confidence intervals. The registered H1 comparisons are OWMAS against SMA, WMA, and EMA; buy-and-hold is retained as an additional reference.

In [ ]:
sharpe_tests = pd.read_csv(RESULTS / 'h1_sharpe_tests.csv')
sharpe_table = sharpe_tests.rename(columns={
    'comparison': 'Comparison',
    'sharpe_owmas': 'OWMAS Sharpe',
    'sharpe_benchmark': 'Benchmark Sharpe',
    'difference': 'Difference',
    'ci_low': '95% CI low',
    'ci_high': '95% CI high',
    'p_value_two_sided': 'p-value',
})[[
    'Comparison', 'OWMAS Sharpe', 'Benchmark Sharpe', 'Difference',
    '95% CI low', '95% CI high', 'p-value'
]]
number_columns = sharpe_table.columns.drop('Comparison')
print(sharpe_table.to_string(index=False, formatters={name: lambda value: f'{value:.4f}' for name in number_columns}))

In [ ]:
registered = sharpe_tests[sharpe_tests['comparison'].isin(['OWMAS - SMA', 'OWMAS - WMA', 'OWMAS - EMA'])]
significant = registered.loc[registered['p_value_two_sided'] < 0.05, 'comparison'].tolist()
if significant:
    print('Significant registered comparisons at the 5% level:', ', '.join(significant))
else:
    print('None of the three registered moving-average comparisons is significant at the 5% level.')
print('H1 is not supported on the full sample. Point-estimate differences should not be treated as evidence of superior Sharpe performance.')

## Full-sample equity curves

In [ ]:
colors = {'OWMAS': '#1f77b4', 'SMA': '#333333', 'WMA': '#777777', 'EMA': '#999999', 'BuyHold': '#b05a2b'}
strategies = ['OWMAS', 'SMA', 'WMA', 'EMA', 'BuyHold']
equity = pd.read_csv(RESULTS / 'equity_curves.csv', parse_dates=['date']).set_index('date')
fig, ax = plt.subplots(figsize=(11, 6))
for name in strategies:
    ax.plot(
        equity.index, equity[name], label=name, color=colors[name],
        linewidth=2.0 if name == 'OWMAS' else 1.15,
    )
ax.set_xlabel('Date')
ax.set_ylabel('Growth of $1')
ax.grid(True, alpha=0.2)
ax.legend(ncol=5, frameon=False, loc='best')
plt.tight_layout()
plt.show()

## Full-sample drawdowns

In [ ]:
drawdowns = pd.read_csv(RESULTS / 'drawdowns.csv', parse_dates=['date']).set_index('date')
fig, ax = plt.subplots(figsize=(11, 6))
for name in strategies:
    ax.plot(
        drawdowns.index, drawdowns[name], label=name, color=colors[name],
        linewidth=2.0 if name == 'OWMAS' else 1.15,
    )
ax.axhline(0.0, color='#222222', linewidth=0.7)
ax.set_xlabel('Date')
ax.set_ylabel('Drawdown')
ax.grid(True, alpha=0.2)
ax.legend(ncol=5, frameon=False, loc='best')
plt.tight_layout()
plt.show()

## Week 9 checkpoint

The preliminary full-sample results are complete. OWMAS has a higher Sharpe point estimate than SMA, is effectively tied with WMA, and is below EMA. None of those three paired Sharpe differences is statistically significant at the 5% level. The result is therefore a valid null finding for H1, not evidence that the experiment failed.

The subperiod, rolling-Sharpe, sensitivity, and holdout results are reserved for the Week 11 Results and Discussion draft.